# Part III: Data Driven Approaches

In this tutorial, we want to explore some data driven methods for image reconstruction. We will consider the limited angle CT problem with additional noise on the sinogram. Our test data will be composed of simple shapes which can be loaded from the ```utils``` module. Namely, the shapes are created via random weighted norm balls in $\ell^p$ norms, i.e., we consider sets

$$ \left\{x\in\mathbb{R}^2: \left(\sum_{i=1}^2 w_i |x_i - m_i|^p\right)^{1/p} < r\right\} $$

where $r>0, w\in\mathbb{R}^2, m\in\mathbb{R}^2$ are sampled randomly.

Let's look at the shapes this produces.

In [ ]:
import matplotlib.pyplot as plt
from utils import random_weighted_norm

img_size = 64 #image size
rwn = random_weighted_norm(img_size=img_size)

# plot all shapes
P = 5
fig, ax = plt.subplots(1, P, figsize=(20,15))
for i in range(P):
    ax[i].imshow(rwn(p=2**i))
    ax[i].set_title('p=' + str(2**i))

# How does our data look like?
Our data is given as the naive recon of a limited angle CT problem. The next cell defines the function that samples a batch of the data. The second output of the function ```get_data``` is the original input (ground truth) that will be used for the supervised learning scheme.

The original image is returned as a  ```torch``` tensor, since this is more convenient for the training later. Furthermore, inputs and outputs into the net will have the shape

$$
B \times C \times N\times N
$$

where

* $B$ denotes the batch size,
* $C$ the number of channels (we always use $C=1$),
* $N$ the image dimension.

We have to be careful when we compare ```numpy``` and ```torch``` objects in the following. If we want to plot a ```torch``` Tensor ```x```, the following steps might be necessary:

* ```x.detach()```: if the tensor ```x``` has a gradient, then we need to detach it first.
* ```x.numpy()```: this converts a ```torch``` tensor, without a gradient, to a ```numpy``` array. Commands like ```plt.imshow``` do this internally.
* ```x[b,c,...]```: ```plt.imshow``` only works for 2D arrays, so we have to select one batch element $b$ and a channel $c$.

We now define a routine that returns our data. Basically, we sample a shape $x$ and then also return

$$R^{-1}(R(x) + \delta)$$

as the input to the net, where $R$ is the Radon transform with limited angles ($0^\circ$ to $90^\circ$), and $\delta$ denotes noise. Here, $R^{-1}$ is not the true inverse, but the naive Radon inversion.

In [ ]:
import torch
import numpy as np
from operators import Radon
from itertools import cycle

num_thetas = 10
min_angle = 0
max_angle = 90
theta = np.linspace(min_angle, max_angle, endpoint = False, num=num_thetas)
R = Radon(theta=theta)

def get_data(batch_size, p, noise_lvl):
    p = cycle([p]) if isinstance(p, (int, float)) else cycle(p)
    
    x_recon = np.zeros((batch_size, 1, img_size, img_size))
    x = np.zeros((batch_size, 1, img_size, img_size))

    # set real images and inputs
    for i in range(batch_size):
        x[i,0,...] = rwn(p=next(p))
        sinogram =  R(x[i,0,...])
        sinogram += np.random.normal(0, noise_lvl, size=sinogram.shape)
        x_recon[i, 0, ...] = R.inverse(sinogram)

    return x_recon, torch.Tensor(x)

## Let's look at our Data!
The below cell samples data from ```get_data```.

In [ ]:
x_recon, x = get_data(5, float('inf'), 0.01)

fig, ax = plt.subplots(1,2, figsize = (10,8))
ax[1].imshow(x_recon[0,0,...])
ax[1].set_title('Naive Recon, input to the net')
ax[0].imshow(x.detach()[0,0,...])
ax[0].set_title('Original')

## Our first approach: Post-processing

Our first approach to employ a data-driven model is so-called **post-processing**. This means, given some data $y^\delta$ we first obtain a (possibly unfavorable) reconstruction with some classical method, e.g., the pseudo-inverse. This gives us $\tilde{x} = A^\dagger y^\delta$. In order to remove artifacts, we then apply some parametrized model $h_\theta$ as a post-processing step. For us, $h_\theta$ will be a trained neural network.

In total, our reconstruction map is then defined as 

$$f_\theta(y^\delta):= h_\theta(A^\dagger y^\delta)$$

## Our neural network: the **U-NET!**

We now define the neural network model, we want to train in the following. Here, we use the celebrated UNet structure from this paper:

<center>
*Ronneberger, O., Fischer, P., & Brox, T. (2015). U-net: Convolutional networks for biomedical image segmentation. In Medical Image Computing and Computer-Assisted Intervention–MICCAI 2015: 18th International Conference, Munich, Germany, October 5-9, 2015, Proceedings, Part III 18 (pp. 234-241). Springer International Publishing.*
</center>


The model architecture is reimplemented (and slightly compressed) in the ```models``` module. We will now load the model and check how many parameters we will train in the following. 

**Spoiler**: Quite a lot.

In [ ]:
from models import UNet
model = UNet()

model_parameters = filter(lambda p: p.requires_grad, model.parameters())
num_params = sum([np.prod(p.size()) for p in model_parameters])

print('Loaded the model with ' + str(num_params) + ' trainable parameters')

# Training the model
In order to train the model, we consider the following minimization problem:

$$
\min_\theta\ \underbrace{\mathbb{E}_{(x,y^\delta)}\left[ \ell(f_\theta(y^\delta), x)\right]}_{:=\mathcal{L}(\theta)}$$

where

* $\theta$ denote the parameters of the neural network $f_\theta$,
* $y^\delta$ is the noisy, badly reconstructed input,
* $x$ is the clean ground truth image,
* $\ell$ is the $\ell^2$ distance, i.e., $\ell(\hat x, x) = \|\hat x - x\|^2$.

In order to solve can use Stochastic Gradient Descent (SGD), which yields the update

$$\theta \gets \theta - \alpha \ \nabla_\theta \left(\sum_{(x,y^\delta) \in B} \ell(f_\theta(y^\delta), x)\right).$$

The difference to standard gradient descent is, that we do not evaluate the true gradient of $\mathcal{L}$. Instead, in each step we sample a finite set $B$ the contains data points. In our implementation, this is realized via the ```get_data``` in each step. 

> **_NOTE:_**  In typical applications one is given a finite training set
> $$\mathcal{T} = \{(x_1, y_1^\delta), \ldots, (x_N, y_N^\delta)\}.$$ 
> In each step of so-called minibatch-SGD one the samples a smaller set $B\subset\mathcal{T}$ which is used as above. In our case here, we can create data on the fly, which allows a slightly non-standard approach.


The parameter $\alpha$ denotes the step size.


## The train step

We now define the train step, that updates the variables $\theta$. The train step typically consists of the following steps:

1. Compute the loss, by evaluating the model and applying ```loss_fct```
2. Compute the gradient: modern libraries like ```PyTorch``` leverage the power of automatic differentiation. As users, that allows us to compute the gradient of a large class of functions, simply by calling the ```backward``` function. Furthermore, the underlying algorithm, so-called **backpropagation** is very efficient (see also the Baur–Strassen theorem).

### A short side note on autograd

In [ ]:
import torch.nn as nn
def loss_fct(x): 
    return (x**2).sum()

par  = nn.Parameter(torch.tensor([0., 1., 2., 3.]))
loss = loss_fct(par)
loss.backward()

print('Gradient after one loss evaluation: ' + str(par.grad))

### Gradient accumulation
It is important to note that gradients are **accumulated**! If we call the function again we get the following:

In [ ]:
loss = loss_fct(par)
loss.backward()

print('Gradient after the second loss evaluation: ' + str(par.grad))

### Zeroing the gradient
In order to correctly compute the gradient, when we call the loss multiple times, we need to zero out the gradients **before** the loss call.

In [ ]:
par.grad.zero_()
loss = loss_fct(par)
loss.backward()

print('Gradient after the second loss evaluation: ' + str(par.grad))

## Back to the train step plan

3. We want to employ a gradient based algorithm, that takes the gradient from above and uses it in the iteration. ```PyTorch``` already implements a large class of optimizers for us, and we simply need to give it the list of all parameters and some hyperparameters. The function ```step``` automatically uses the gradient of the parameters and performs the step. The optimizer also provides a function to zero out the gradients of all parameters involved.

In [ ]:
def loss_fct(x, z):
    return (x**2).sum() * z

par1  = nn.Parameter(torch.tensor([0., 1., 2., 3.]))
par2  = nn.Parameter(torch.tensor([4.]))

opt = torch.optim.SGD([par1, par2], lr=1)

loss = loss_fct(par1,par2)
loss.backward()
opt.step()

print('Gradients: ' + str(par1.grad) + ', ' + str(par2.grad))
print('Variables at new step: ' + str(par1.data) + ', ' + str(par2.data))

opt.zero_grad()

print('Gradients after zero grad: ' + str(par1.grad) + ', ' + str(par2.grad))

loss = loss_fct(par1,par2)
loss.backward()
opt.step()

print('Gradients after second step: ' + str(par1.grad) + ', ' + str(par2.grad))
print('Variables at second step: ' + str(par1.data) + ', ' + str(par2.data))

### &#128221; <span style="color:darkorange"> Task 3.1 </span>
#### Define the train step

Your task is to now implement the train step based on the insights from above.

1. Define an appropriate loss function, we want to employ the following criterion between two reconstructions: $\ell(\hat x, x) = \|\hat x - x\|^2$
2. Define the optimizer and choose an appropriate learning rate. Instead of SGD on typically employs the ADAM optimizer (Kingma, Diederik P. "Adam: A method for stochastic optimization.). For Adam you typically use a smaller learning rate.
3. Use these functions to define the train step.

In [ ]:
loss_fct = nn.MSELoss()
opt = torch.optim.Adam(model.parameters(), lr=0.0005)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=30)

In [ ]:
def train_step(model, get_data, loss_fct, opt, scheduler, batch_size=5, p=2, noise_lvl=0.01):
    opt.zero_grad() # zero out gradients from previous step

    x_recon, x = get_data(batch_size, p, noise_lvl) # get data
    x_model = model(x_recon) # compute the model
    loss = loss_fct(x_model, x) # compute the loss
    loss.backward() # compute the gradients
    opt.step() # make a step of the optimizer
    scheduler.step(loss) # make a scheduler step

    # additonal computations for tracking
    loss1 = loss.item()
    loss2 = loss_fct(torch.Tensor(x_recon), torch.Tensor(x)).item()

    print(30*'-')
    print('Iteration: ' + str(num_it))
    print('Current Model Loss:' + str(loss1))
    print('Recon Loss:' + str(loss2))
    for param_group in opt.param_groups:
        print('Current lr:' + str(param_group['lr']))

### &#128221; <span style="color:darkorange"> Task 3.2 </span>
#### Execute the training

Your next task is to train the network. Please specify the following

* ```train_p```: the values of $p$ to be used to train the network on
* ```batch_size```: the batch size
* ```max_it```: the maximal number of iteration step
* ```noise_lvl```: the noise level

Within your group you should choose different train shapes. After training you can evaluate your result with the cell below.

In [ ]:
from tqdm.notebook import trange

train_p = [2]
batch_size = 5
max_it = 400
noise_lvl = 0.02

for num_it in trange(max_it):
    train_step(
        model, get_data, loss_fct, opt, scheduler, 
        batch_size=batch_size, p=train_p, noise_lvl = noise_lvl
    )

## Saving and loading the models
If you want, you can save or load models with the functions below.

In [ ]:
import datetime
save_model = True
name = 'unet_post_p_' + str(train_p) + '_' + str(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
if save_model:
    torch.save(model.state_dict(), name + '.pt')

In [ ]:
## load a model
load_model = False
if load_model:
    model.load_state_dict(torch.load('circle.pt', weights_only=True))

### &#128221; <span style="color:darkorange"> Task 3.3 </span>
#### Evaluating the results

The cell below allows you to test the performance of your trained model.

Your task is to try out different shapes, noise levels and angle specification and evaluate the performance of your model :)

In [ ]:
from utils import get_phantom
import ipywidgets as widgets
from ipywidgets import interactive
from IPython.display import display
from skimage.metrics import structural_similarity as ssim
import random 

img_size_test=64
im_kwargs = {'cmap':'bone'}

def get_data_eval(p, noise_lvl, angle, num_samples = 1):
    theta = np.linspace(angle[0],angle[1], endpoint = False, num=num_thetas)
    R = Radon(theta=theta)

    X, X_recon = [], []
    for _ in range(num_samples):
        x = rwn(p=p)
        sinogram = R(x)
        sinogram += np.random.normal(0, noise_lvl, size=sinogram.shape)
        x_recon = R.inv(sinogram)
        X.append(x)
        X_recon.append(x_recon)
    x, x_recon = np.stack(X,axis=0), np.stack(X_recon,axis=0)
        
    x_model = model(x_recon).detach().numpy()
    return x.squeeze(), x_recon.squeeze(), x_model.squeeze()

def plot_result(p, noise_lvl, angle, seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    x, x_recon, x_model = get_data_eval(p, noise_lvl, angle)
    
    fig, ax = plt.subplots(1,3, figsize = (20,15))

    for i, (z, title) in enumerate([(x,'Ground truth'), (x_recon, 'Naive Recon'), (x_model,'Network Recon')]):  
        ax[i].imshow(z, **im_kwargs)
        ax[i].set_title(
            title + ', error: '+ str(round(np.linalg.norm(x - z), 4)) + '\n' +
            'SSIM: ' + str(ssim(x,z, data_range=x.max() - x.min())))  
        ax[0].set_title('p=' +str(p))

p_slider = widgets.FloatSlider(min = 0.5, max = 25, step = 0.5, value = 2, continuous_update = False)
n_slider = widgets.FloatSlider(min = 0.0, max = .1, step = 0.001, value = 0.01, continuous_update = False)
a_slider = widgets.FloatRangeSlider(value=[0, 90],min=0,max=180,step=10,continuous_update=False)
s_slider = widgets.IntSlider(min=0, max=100, step=1, value=0, continuous_update=False)

interactive_plot = interactive(plot_result, p = p_slider, angle=a_slider, noise_lvl=n_slider, seed=s_slider)
display(interactive_plot)

## Quantitativ results

In [ ]:
num_samples = 50

def set_error(E, x, x_recon, x_model):
    ks = ['l2_recon', 'l2_model', 'ssim_recon', 'ssim_model']
    for k in ks: 
        if not k in E.keys(): E[k] = []
        z = x_recon if k.split('_')[-1] == 'recon' else x_model
        if k.split('_')[0] == 'l2':
            E[k].append(np.linalg.norm(x - z, axis=-1).mean())
        else:
            e = 0
            for i in range(x.shape[0]): 
                e += ssim(x[i,...], z[i,...], data_range=1.)
            E[k].append(e/x.shape[0])

# loop over p
angle = [0,90]
noise_lvl = 0.02
Ep = {}
ps = np.arange(1,10)
for p in ps:
    x, x_recon, x_model = get_data_eval(p, noise_lvl, angle, num_samples = num_samples)
    set_error(Ep, x, x_recon, x_model)

# loop over different end angles
p = 2
Ea = {}
angles = [60, 90, 120, 150, 180]
for a in angles:
    angle[1] = a
    x, x_recon, x_model = get_data_eval(p, noise_lvl, angle, num_samples = num_samples)
    set_error(Ea, x, x_recon, x_model)

# loop over different noise levels
p = 2
angle = [0,90]
En = {}
noise_lvls = np.linspace(0,0.2, 10)
for n in noise_lvls:
    x, x_recon, x_model = get_data_eval(p, n, angle, num_samples = num_samples)
    set_error(En, x, x_recon, x_model)

In [ ]:
fig, ax = plt.subplots(2,3, figsize=(15,10))
for i, (v, E, t) in enumerate([(ps,Ep, 'p'), (angles, Ea, 'End angle'), (noise_lvls, En, 'noise level')]):
    for k in E.keys():
        m = 0 if  k.split('_')[0] == 'l2' else 1
        ax[m,i].plot(v, E[k], label=k)
        ax[m,i].set_xlabel(t)
        ax[m,i].legend()
plt.legend()

# Second approach: plug-and-play!

A popular way to combine model and data driven approaches are so-called plug-and-play (PnP) methods. The starting point is a variational minimization problem,

$$\min_x \frac{1}{2}\, \|Ax - y\|^2 + \lambda J(x).$$

This problem can be solved via prox-based methods, for example an ADMM update scheme

$$
\begin{align*}
x &\gets \operatorname*{arg min}_{x}\ \frac{1}{2}\, \|Ax - y\|^2 + \frac{\rho}{2} \|{v - u}\|^2,\\
v &\gets \operatorname*{arg min}_{v}\ \lambda J(v) + \frac{\rho}{2} \|{v - (x + u)}\|^2,\\
u &\gets u + x - v.
\end{align*}
$$

Here, the first line can be solved can be solved with a linear solver (e.g. the cg iteration) and the last line ist explicit. The second line is in fact the prox operator of $J$ since 

$$\operatorname{prox}_{\lambda/\rho\ J}(x+u) =  \operatorname*{arg min}_{v}\ \lambda J(v) + \frac{\rho}{2} \|{v - (x + u)}\|^2.$$

Evaluating this prox can be complicated and relies on a possibly hand-crafted functional $J$. The idea of PnP methods consists of replacing a prox step of this kind by an arbitratry map $D$. I.e. the iteration takes the form

$$
\begin{align*}
x &\gets \operatorname*{arg min}_{x}\ \frac{1}{2}\, \|Ax - y\|^2 + \frac{\rho}{2} \|{v - u}\|^2,\\
v &\gets D_\lambda(x + u),\\
u &\gets u + x - v.
\end{align*}
$$





## Observation: prox solves denoising!

The prox operator is defined as

$$\operatorname{prox}_{1/\rho J}(x) =  \operatorname*{arg min}_{v}\  J(v) + \frac{\rho}{2} \|{v - x}\|^2.$$

This can be interpreted as the solution map for the inverse problem

$$ y = x+ \varepsilon,$$

with noise $\varepsilon\sim\mathcal{N}(0, \rho)$.

**Idea**: Replace the prox-operator with a denoiser!

## Training the denoiser

We want to try out the PnP approaches from above. To do so we first train a denoiser $D$. Here, we use the same setup as before, we just have to change the ```get_data``` function.

### &#128221; <span style="color:darkorange"> Task 3.4 </span>
#### Train again

Redefine the ```get_data``` function, such that it produces data for the denoising task.

In [ ]:
def get_data(batch_size, p, noise_lvl):
    x_recon = np.zeros((batch_size, 1, img_size, img_size))
    x = np.zeros((batch_size, 1, img_size, img_size))

    for i in range(batch_size):
        x[i,0,...] = rwn(p)
        x_recon[i, 0, ...] = x[i,0,...] + np.random.normal(0, noise_lvl, size=x[i,0,...].shape)
    return x_recon, torch.Tensor(x)

## Look at the data again

The data looks slightly different now.

In [ ]:
x_recon, x = get_data(5, 2, 0.2)

fig, ax = plt.subplots(1,2, figsize = (10,8))
ax[1].imshow(x_recon[0,0,...], **im_kwargs)
ax[1].set_title('Noisy version')
ax[0].imshow(x.detach()[0,0,...], **im_kwargs)
ax[0].set_title('Original');

### &#128221; <span style="color:darkorange"> Task 3.5 </span>
#### Train again

With the same set-up as before, you can now train the denoising model. Define a new model and optimizer, and train the denoiser.

In [ ]:
denoiser  = UNet()
opt       = torch.optim.Adam(denoiser.parameters(), lr=0.001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, patience=30)

In [ ]:
train_p = [2]
batch_size = 5
max_it = 150
noise_lvl = 0.3

for num_it in range(max_it):
    train_step(
        denoiser, get_data, loss_fct, opt, scheduler, 
        batch_size=batch_size, p=train_p, noise_lvl = noise_lvl
    )

## Saving the Denoiser
As before, if you want you can save or load a denoiser.

In [ ]:
import datetime
save_model = True
name = 'denoiser_' + str(train_p) + '_' + str(datetime.datetime.now().strftime("%Y%m%d%H%M%S"))
if save_model:
    torch.save(denoiser.state_dict(), name + '.pt')

In [ ]:
load_model = False
if load_model:
    denoiser.load_state_dict(torch.load('denoiser.pt', weights_only=True))

## How well does it work?
The cell below shows the denoising performance of the trained network.

In [ ]:
x_recon, x = get_data(5, 2, noise_lvl)

fig, ax = plt.subplots(1,3, figsize=(9,3))
ax[2].imshow(denoiser(x_recon).detach()[0,0,...], **im_kwargs)
ax[2].set_title('Model Recon')
ax[1].imshow(x_recon[0,0,...], **im_kwargs)
ax[1].set_title('Naive Recon')
ax[0].imshow(x.detach()[0,0,...], **im_kwargs)
ax[0].set_title('Original');

## Defining the PnP prox substitute

Our original goal was to substitute the prox in the admm itertion. In the module ```optimizer``` the optimizer ```admm``` is defined. It has the following signature for initialization:

```admm(R, x0, Rx, rho=0.4, lamda=1., verbosity=0, prox=model_prox, max_it=35, max_inner_it=1)```

where

* ```R``` is the linear operator, i.e. the Radon trafo
* ```x0``` is the inital guess
* ```rho``` is an iteration parameter
* ```lamda``` scales the influence of the prox
* ```prox``` defines the prox mapping
* ```max_it``` determines the umber of steps
* ```max_inner_it``` determines the number of inner iterations

We just have to define the prox maping

In [ ]:
from optimizers import admm

def model_prox(x, lamda):
    lamda = max(min(lamda, 1.),0)
    return (1-lamda) * x + lamda * denoiser(x).detach().numpy()[0,0,...]

def pnp_recon(R, y):
    x0 = R.inv(y)
    pnpadmm = admm(R, x0, y, rho=0.4, lamda=1., 
                   verbosity=0, prox=model_prox, max_it=30, max_inner_it=1)
    pnpadmm.solve(use_tqdm=True)
    return pnpadmm.x

### &#128221; <span style="color:darkorange"> Task 3.6 </span>
#### Test the performance on the CT problem

With the following cell you can now test how well the denoiser and the PnP-ADMM iteration performs on the CT reconstruction task. How does the noise level influence the performance?

In [ ]:
def get_data_eval_pnp(p, noise_lvl, angle, num_samples = 1):
    theta = np.linspace(angle[0],angle[1], endpoint = False, num=num_thetas)
    R = Radon(theta=theta)

    X, X_recon, X_PnP = [], [], []
    for _ in range(num_samples):
        x = rwn(p=p)
        sinogram = R(x)
        sinogram += np.random.normal(0, noise_lvl, size=sinogram.shape)
        x_recon = R.inv(sinogram)
        X.append(x)
        X_recon.append(x_recon)
        X_PnP.append(pnp_recon(R, sinogram))
    x, x_recon, x_pnp = np.stack(X,axis=0), np.stack(X_recon,axis=0), np.stack(X_PnP,axis=0)
    x_model = model(x_recon).detach().numpy()
    return x.squeeze(), x_recon.squeeze(), x_pnp.squeeze(), x_model.squeeze()

def plot_result(p, noise_lvl, angle, seed):
    torch.manual_seed(seed)
    random.seed(seed)
    np.random.seed(seed)
    x, x_recon, x_pnp, x_model = get_data_eval_pnp(p, noise_lvl, angle)
    
    fig, ax = plt.subplots(1,4, figsize = (20,15))

    for i, (z, title) in enumerate([(x,'Ground truth'), (x_recon, 'Naive Recon'), (x_pnp,'PnP Recon'),
                                   (x_model,'Post Processing Recon')]):  
        ax[i].imshow(z, **im_kwargs)
        ax[i].set_title(title + 
                        ', error: '+ str(round(np.linalg.norm(x - z), 4)) + '\n' +
                        'SSIM: ' + str(ssim(x,z, data_range=x.max() - x.min())))
    ax[0].set_title('p=' +str(p))


p_slider = widgets.FloatSlider(min = 0.5, max = 25, step = 0.5, value = 2, continuous_update = False)
n_slider = widgets.FloatSlider(min = 0.0, max = .1, step = 0.001, value = 0.01, continuous_update = False)
a_slider = widgets.FloatRangeSlider(value=[0, 90],min=0,max=180,step=10,continuous_update=False)
s_slider = widgets.IntSlider(min=0, max=100, step=1, value=0, continuous_update=False)

interactive_plot = interactive(plot_result, p = p_slider, angle=a_slider, noise_lvl=n_slider, seed=s_slider)
display(interactive_plot)

## Caveats
* Convergence of ADMM is not directly clear, when we replace the prox with an arbitrary map.
* The choice of noise level in the denoiser is an intricate detail. In our case we only trained one denoiser on a hand-picked level.